In [1]:
import requests
import json
import base64

# ========== 核心 Agent ==========
class ERPAgent:
    def __init__(self):
        self.round = 1
        self.history = []
        self.competitor_history = []  # 记录对手广告费
        self.data = {
            "现金": 100,
            "固定资产": 50,
            "库存": 0,
            "贷款": 0,
            "所有者权益": 100,
            "研发进度": {"P1": "完成", "P2": "未开始"},
            "广告费": 0,
            "生产线数量": 1,
            "每季度固定支出": 10
        }

    def calculate_equity(self):
        equity = (self.data["现金"] + self.data["固定资产"]
                  + self.data["库存"] - self.data["贷款"])
        self.data["所有者权益"] = equity
        return equity

    def update_data(self, updates: dict):
        self.data.update(updates)
        self.calculate_equity()
        print("数据已更新，当前所有者权益：" + str(self.data["所有者权益"]))

    def show_status(self):
        print("\n" + "="*45)
        print("第 " + str(self.round) + " 轮财务状态")
        print("="*45)
        for k, v in self.data.items():
            print("  " + str(k) + ": " + str(v))
        print("="*45 + "\n")

    # ===== 功能1：现金流死线预测 =====
    def cash_flow_deadline(self):
        cash = self.data["现金"]
        fixed_cost = self.data["每季度固定支出"]
        loan = self.data["贷款"]
        loan_interest = loan * 0.05  # 假设5%利息
        total_cost_per_round = fixed_cost + loan_interest
        if total_cost_per_round <= 0:
            seasons = 999
        else:
            seasons = int(cash / total_cost_per_round)
        print("\n现金流预警：")
        print("  当前现金：" + str(cash))
        print("  每季度支出（含利息）：" + str(round(total_cost_per_round, 2)))
        print("  若拿不到订单，还能撑：" + str(seasons) + " 个季度")
        if seasons <= 2:
            print("  !!!高危警告：现金流紧张，谨慎贷款和广告!!!")
        return seasons

    # ===== 功能2：竞标压力分析 =====
    def competitor_analysis(self, competitor_ads: list):
        self.competitor_history.extend(competitor_ads)
        if len(self.competitor_history) == 0:
            print("暂无对手数据")
            return
        avg = sum(self.competitor_history) / len(self.competitor_history)
        max_ad = max(self.competitor_history)
        print("\n对手广告费分析：")
        print("  历史平均：" + str(round(avg, 1)))
        print("  历史最高：" + str(max_ad))
        print("  建议本轮广告费区间：" + str(round(avg * 0.9, 1)) + " ~ " + str(round(avg * 1.2, 1)))

    # ===== 功能3：视觉识别截图数据（Qwen3-VL）=====
    def extract_from_image(self, image_path: str):
        with open(image_path, "rb") as f:
            img_data = base64.b64encode(f.read()).decode("utf-8")
        prompt = "请提取图中所有订单的产品名、数量、单价，输出为Python字典格式。"
        resp = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "qwen3-vl:8b",
                "prompt": prompt,
                "images": [img_data],
                "stream": False
            },
            timeout=120
        )
        result = resp.json()["response"]
        print("\n图像识别结果：")
        print(result)
        return result

    # ===== 功能4：主决策建议（Qwen3）=====
    def get_strategic_advice(self, market_info: str):
        self.show_status()
        seasons_left = self.cash_flow_deadline()

        prompt = (
            "你是ERP比赛CEO顾问，给出第" + str(self.round) + "轮决策建议。\n"
            "财务数据：" + str(self.data) + "\n"
            "市场信息：" + market_info + "\n"
            "历史决策：" + str(self.history[-3:] if self.history else "暂无") + "\n"
            "现金流警告：还能撑" + str(seasons_left) + "季度\n"
            "核心规则：最终以所有者权益高低决定胜负，贷款有利息风险。\n\n"
            "请给出：\n"
            "1. 广告费建议金额及理由\n"
            "2. 是否贷款，贷多少\n"
            "3. 生产线调整建议\n"
            "4. 研发投入建议\n"
            "5. 本轮最优先事项"
        )

        print("AI正在分析，请稍候...")
        resp = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": "qwen3:8b", "prompt": prompt, "stream": False},
            timeout=180
        )
        advice = resp.json()["response"]
        self.history.append({
            "轮次": self.round,
            "市场": market_info,
            "权益": self.data["所有者权益"]
        })
        self.round += 1
        return advice


# ========== 启动 ==========
my_agent = ERPAgent()

# 输入对手上轮广告费（没有就留空列表）
my_agent.competitor_analysis([15, 20, 18])

# 输入本轮市场信息，运行获取建议
market_info = "本轮市场需求旺盛，竞争对手加大广告投入，原材料价格上涨10%"
advice = my_agent.get_strategic_advice(market_info)
print("\nAI建议：\n")
print(advice)


对手广告费分析：
  历史平均：17.7
  历史最高：20
  建议本轮广告费区间：15.9 ~ 21.2

第 1 轮财务状态
  现金: 100
  固定资产: 50
  库存: 0
  贷款: 0
  所有者权益: 100
  研发进度: {'P1': '完成', 'P2': '未开始'}
  广告费: 0
  生产线数量: 1
  每季度固定支出: 10


现金流预警：
  当前现金：100
  每季度支出（含利息）：10.0
  若拿不到订单，还能撑：10 个季度
AI正在分析，请稍候...

AI建议：

### 1. **广告费建议金额及理由**  
**建议金额：30**  
**理由**：  
- 市场需求旺盛但竞争对手加大广告投入，需抢占市场份额。  
- 原材料价格上涨10%会压缩利润空间，需通过广告费提升销量以抵消成本压力。  
- 广告费投入需平衡短期收益与现金流风险。当前现金100，每季度固定支出10，贷款利息风险高，建议将广告费控制在**30**（占现金流的30%），既能快速扩大市场影响力，又不会过度消耗现金流。  

---

### 2. **是否贷款，贷多少**  
**建议：不贷款**  
**理由**：  
- 贷款需支付利息（规则未明确利率，但存在风险），可能侵蚀所有者权益。  
- 当前现金流可支撑10个季度，无需额外融资。  
- 若未来需扩大产能或研发，可优先通过提升销量和利润积累资金，避免债务风险。  

---

### 3. **生产线调整建议**  
**建议：暂不调整**  
**理由**：  
- 当前生产线数量为1，市场需求旺盛，需优先通过广告费提升销量以充分利用现有产能。  
- 原材料价格上涨可能推高生产成本，若盲目扩建生产线会进一步增加固定成本压力。  
- 若后续季度销量持续增长且现金流充足，可考虑新增生产线。  

---

### 4. **研发投入建议**  
**建议：投入P2研发（金额待定）**  
**理由**：  
- 研发进度P1已完成，P2是关键突破点。若P2能带来技术优势（如降低成本或提升产品竞争力），需尽早投入。  
- 当前所有者权益为100，可考虑投入**10-20**（占现金流的10%-20%）启动P2研发，但需结合广告费预算调整。  
- 若研发资金紧张，可优先

In [5]:
# 基于“赛项 A 卷”官方数据的参数初始化
params = {
    # --- 初始财务状态 ---
    "初始现金": 100000,           # 现金 100,000
    "初始所有者权益": 700000,      # 所有者权益合计 700,000
    "原材料价值": 600000,         # R1-R4 各 300 个，合计 600,000
    
    # --- 运营硬性规则 ---
    "预算使用率下限": 0.8,         # 低于 80% 扣 10000 分
    "预算使用率上限": 1.2,         # 高于 120% 扣 10000 分
    "所得税率": 0.25,             # 25%
    "违约扣款比例": 0.40,          # 违约扣 40%
    
    # --- 费用类 ---
    "每季管理费": 10000,          # 固定管理费 10,000
    "折旧年限_传统": 1,            # 传统线 1 年折旧
    "折旧年限_全自动": 3,          # 全自动 3 年折旧
    
    # --- 碳中和 ---
    "初始碳排放量": 100000,        # 100,000 吨
    "碳中和单价": 20              # 20 元/吨
}

In [7]:
import pandas as pd
import os

# 正确获取路径：使用 OneDrive/Desktop
desktop = os.path.join(os.path.expanduser('~'), 'OneDrive', 'Desktop')
file_name = '经销商订单明细 (1).xls'
file_path = os.path.join(desktop, file_name)

# 验证文件是否存在
if not os.path.exists(file_path):
    print(f"❌ 文件未找到: {file_path}")
else:
    # 读取Excel文件
    df = pd.read_excel(file_path, engine='xlrd')
    print("✅ 读取成功！")
    print(f"数据行数: {len(df)}")


✅ 读取成功！
数据行数: 63


In [10]:
import pandas as pd
import os

# 文件真实路径（根据你提供的位置）
file_path = r"C:\Users\Microsoft\OneDrive\Desktop\经销商订单明细 (1).xls"

# 检查文件是否存在
if not os.path.exists(file_path):
    print(f"❌ 文件不存在！请检查路径：{file_path}")
else:
    # 读取 Excel 的 sheet1
    df_orders = pd.read_excel(file_path, sheet_name='sheet1')
    print("✅ 读取成功！")
    print(f"数据行数：{len(df_orders)}")

    # 测试函数（根据你的需求调整）
    def get_best_order(year, quarter):
        current = df_orders[(df_orders['年份'] == year) & (df_orders['季度'] == quarter)].copy()
        current['估算毛利'] = current['供应商参考价格(元)'] - 2000 - (current['供应商参考价格(元)'] * 0.08)
        return current.sort_values(by='估算毛利', ascending=False).head(5)

    # 测试
    print(get_best_order(1, 2))


✅ 读取成功！
数据行数：63
   年份  季度  编号    市场  产品  特性  供应商参考价格(元)   数量  交货期(季)  账期(季)       认证     估算毛利
1   1   2   2  国内市场  P1  T2        1249  471       4      2  ISO9000  -850.92
3   1   2   4  国内市场  P1  T3        1130  756       4      4  ISO9000  -960.40
2   1   2   3  国内市场  P1  T2        1118  789       3      3  ISO9000  -971.44
4   1   2   5  国内市场  P1  T3        1070  756       4      1  ISO9000 -1015.60
0   1   2   1  国内市场  P1  T1        1057  588       4      1          -1027.56


In [12]:
# 修复后完整代码
import pandas as pd
import os

# 正确获取路径（含 OneDrive）
desktop = os.path.join(os.path.expanduser('~'), 'OneDrive', 'Desktop')
file_name = "经销商订单明细 (1).xls"
file_path = os.path.join(desktop, file_name)

# 检查文件是否存在
if not os.path.exists(file_path):
    print(f"❌ 文件未找到！请检查路径：{file_path}")
else:
    # 读取 Excel 文件
    df = pd.read_excel(file_path, engine='xlrd')
    print("✅ 读取成功！")
    print(f"数据行数：{len(df)}")

    # 测试函数
    def get_best_order(year, quarter):
        current = df[(df['年份'] == year) & (df['季度'] == quarter)].copy()
        current['估算毛利'] = current['供应商参考价格(元)'] - 2000 - (current['供应商参考价格(元)'] * 0.08)
        return current.sort_values(by='估算毛利', ascending=False).head(5)

    print(get_best_order(1, 2))


✅ 读取成功！
数据行数：63
   年份  季度  编号    市场  产品  特性  供应商参考价格(元)   数量  交货期(季)  账期(季)       认证     估算毛利
1   1   2   2  国内市场  P1  T2        1249  471       4      2  ISO9000  -850.92
3   1   2   4  国内市场  P1  T3        1130  756       4      4  ISO9000  -960.40
2   1   2   3  国内市场  P1  T2        1118  789       3      3  ISO9000  -971.44
4   1   2   5  国内市场  P1  T3        1070  756       4      1  ISO9000 -1015.60
0   1   2   1  国内市场  P1  T1        1057  588       4      1          -1027.56


In [16]:
import openai

# 1. 建立与本地 Ollama 的连接
client = openai.OpenAI(
    base_url='http://localhost:11434/v1/',  # 注意：最后这个 /v1/ 绝对不能漏掉
    api_key='ollama',                       # 随便填，但绝不能是空的
)

# 2. 呼叫你的 qwen 大脑
try:
    response = client.chat.completions.create(
        model="qwen3-vl:8b ",             # 这里的名字必须和你黑框框里的一模一样
        messages=[
            {"role": "user", "content": "你好，我是沙盘比赛企业的CEO。测试通讯，收到请用中文回复一句简短的确认。"}
        ]
    )
    # 3. 打印模型回复
    print("🤖 Agent 回复：")
    print(response.choices[0].message.content)

except Exception as e:
    print(f"❌ 还是报错了，错误信息是：\n{e}")

🤖 Agent 回复：
收到，测试通讯确认。
